# Guia da Aula 02 — Métodos Baseados em Valores

**Disciplina:** Modelos de Aprendizagem por Reforço  
**Aula:** 02 — Value-Based Methods  
**Formato:** Guia de navegação e validação de ambiente

## Informações do notebook

| Item | Detalhe |
|---|---|
| **Aula** | Aula 02 — Métodos Baseados em Valores |
| **Notebook** | 00 — Guia de Navegação |
| **Tempo de leitura** | 8 min |
| **Tempo de execução** | 2 min |

### Competências para o Desafio Final

- Identificar qual método value-based é adequado a cada cenário do Desafio Final.
- Validar o ambiente de execução e confirmar a disponibilidade das dependências (incluindo box2d).
- Compreender a progressão DP → MC → TD → Q-Learning → SARSA → DQN antes de iniciar o estudo.

### Recapitulando

A Aula 01 formalizou os três paradigmas de ML — supervisionado, não supervisionado e RL — e implementou um bandit ε-greedy no contexto MovieLens.

A limitação central que ficou aberta: o bandit não tem estados que mudam e não considera consequências futuras.

A Aula 02 resolve isso com os primeiros algoritmos que tratam MDPs completos, com estados variáveis e propagação de valor ao longo do tempo.

In [4]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import rl_utils
rl_utils.info_versoes()
rl_utils.configurar_matplotlib()
rl_utils.definir_seeds(42)

Biblioteca           Versão
--------------------------------
gymnasium            1.0.0
torch                2.11.0+cu130
numpy                2.4.2
matplotlib           3.10.8
pandas               3.0.1
scikit-learn         1.8.0


## Bloco 1 — Contexto e pergunta central



A Aula 01 apresentou três paradigmas de aprendizado — supervisionado, não supervisionado e por reforço. Esta aula se dedica inteiramente ao RL e percorre uma família de métodos que responde a uma pergunta fundamental:

> **"Como um agente aprende a atribuir valor a estados e ações para tomar melhores decisões?"**

O fio condutor é a progressão dos métodos baseados em valor:

> intuição de valor → Equação de Bellman → Programação Dinâmica → Monte Carlo → TD(0) → Q-Learning → SARSA → DQN

Cada notebook representa um passo nessa progressão.

## Bloco 2 — Mapa da aula e ordem de execução

Execute os notebooks **na ordem numérica** (00 → 05). Cada notebook é autocontido, mas a progressão conceitual faz mais sentido quando seguida em ordem.

| Notebook | Tema | Ambiente | Seções do plano |
|---|---|---|---|
| `00_guia_aula02.ipynb` | Guia de navegação + validação de ambiente | — | — |
| `01_valor_bellman_dp.ipynb` | Intuição de valor, Equação de Bellman, Programação Dinâmica | GridWorld 4×4 (custom) | 2.1, 2.2, 2.3 |
| `02_mc_td_qlearning.ipynb` | Monte Carlo, TD(0), Q-Learning | FrozenLake-v1 | 2.4, 2.5 |
| `03_sarsa_onpolicy.ipynb` | SARSA e comparação on-policy vs off-policy | Taxi-v3 | 2.6 |
| `04_dqn_aproximacao.ipynb` | DQN, melhorias (Double, Dueling, PER) e limitações | LunarLander-v3 | 2.7, 2.8, 2.9 |
| `05_comparativo_final.ipynb` | Comparação entre todos os métodos + transição Aula 03 | FrozenLake + Taxi | 2.1–2.9 |

> **Atenção:** O Notebook 04 exige `gymnasium[box2d]` e a dependência de sistema `swig`. Veja as instruções na próxima seção.

## Como executar os notebooks

**Ambiente Python:** Python ≥ 3.9. Ative o ambiente virtual e instale as dependências:

    source .venv/bin/activate
    pip install -r requirements.txt
    jupyter lab

**Dependência adicional para o Notebook 04 (DQN / LunarLander-v3):**

    pip install "gymnasium[box2d]"
    # Linux: sudo apt-get install swig

**Reprodutibilidade:** semente aleatória fixada em todos os notebooks (`SEED = 42`). Os resultados podem variar ligeiramente entre versões de bibliotecas.

| Notebook | Tempo estimado (CPU) |
|---|---|
| 00 — Guia | < 1 min |
| 01 — Bellman / DP | 1–2 min |
| 02 — MC / TD / Q-Learning | 2–4 min |
| 03 — SARSA | 2–4 min |
| 04 — DQN | 5–10 min |
| 05 — Comparativo | 5–10 min |

In [5]:
# Instalação opcional das dependências principais
# %pip install numpy matplotlib gymnasium torch

# Para o Notebook 04 (DQN / LunarLander), instalar também:
# %pip install "gymnasium[box2d]"
# No Linux, antes: !apt-get install -y swig

In [6]:
import sys
import importlib

print(f"Python: {sys.version}")
print()

bibliotecas = {
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "gymnasium": "gymnasium",
    "torch": "torch",
}

for nome, modulo in bibliotecas.items():
    try:
        m = importlib.import_module(modulo)
        versao = getattr(m, "__version__", "versão não disponível")
        print(f"  {nome}: {versao}")
    except ImportError:
        print(f"  {nome}: NÃO INSTALADO")

# Verificação específica de gymnasium[box2d] (necessário apenas para o Notebook 04)
try:
    import gymnasium.envs.box2d  # noqa: F401
    print("  gymnasium[box2d]: disponível")
except (ImportError, Exception):
    print("  gymnasium[box2d]: não disponível (necessário apenas para o Notebook 04)")

print()
print("Ambiente validado. Execute os notebooks na ordem: 00 → 01 → 02 → 03 → 04 → 05")

Python: 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]

  numpy: 2.4.2
  matplotlib: 3.10.8
  gymnasium: 1.0.0
  torch: 2.11.0+cu130
  gymnasium[box2d]: disponível

Ambiente validado. Execute os notebooks na ordem: 00 → 01 → 02 → 03 → 04 → 05


/home/drl/fiap/rl-ai-scientist/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


## Bloco 3 — Leitura dos resultados da validação

A célula acima imprime as versões de cada biblioteca. Se `gymnasium[box2d]` aparecer como "não disponível", os Notebooks 00 a 03 e 05 continuarão funcionando normalmente — apenas o Notebook 04 (DQN/LunarLander) exige essa dependência adicional.

Caso `torch` não esteja instalado, o Notebook 04 também ficará indisponível. Todos os demais notebooks usam apenas `numpy` e `matplotlib`.

## Bloco 4 — Interpretação pedagógica da progressão

Cada notebook responde a uma versão mais complexa da mesma pergunta central. A progressão foi projetada para que o aluno perceba o que muda quando cada limitação anterior é superada:

- **Notebook 01** mostra como calcular valor quando o modelo do ambiente está completamente disponível (Programação Dinâmica).
- **Notebook 02** introduz o aprendizado sem modelo: Monte Carlo usa retornos completos; TD(0) usa bootstrapping.
- **Notebook 03** torna visível a diferença entre aprender sobre o comportamento atual (on-policy / SARSA) e o comportamento ótimo (off-policy / Q-Learning).
- **Notebook 04** substitui a tabela Q por uma rede neural quando o espaço de estados deixa de ser discreto e pequeno.
- **Notebook 05** consolida a comparação entre todos os métodos e aponta para a Aula 03 (Policy-Based Methods).

## Bloco 5 — Limites deste guia e próximo passo

Este notebook não ensina nenhum algoritmo; sua função é orientar a navegação e confirmar que o ambiente está pronto.

O aprendizado começa no **Notebook 01 (`01_valor_bellman_dp.ipynb`)**, com a intuição de valor e a Equação de Bellman aplicada a um GridWorld simples — o ponto de partida mais concreto para responder à pergunta central desta aula.

## Conexão com o problema de recomendação MovieLens

Os algoritmos desta aula — de Programação Dinâmica ao DQN — podem ser aplicados diretamente ao contexto de recomendação introduzido na Aula 01:

| Algoritmo | Aplicação em recomendação MovieLens |
|---|---|
| **Programação Dinâmica** | Planejamento offline de sequência de recomendações, quando as probabilidades de transição entre estados de usuário são conhecidas |
| **Q-Learning / SARSA** | Aprender Q(estado_usuário, filme) — estimar qual filme tem maior valor esperado para cada perfil, sem modelo explícito |
| **DQN** | Usar embeddings de usuário/filme como estado contínuo — escala para catálogos grandes onde o espaço de estados tabular é inviável |

A nota MovieLens (`gostou = nota ≥ 4`) é a recompensa natural do MDP. O estado pode ser o cluster do usuário (Aula 01, NB03) ou um vetor de preferências. As ações são os filmes candidatos. O **Desafio Final** conecta esses componentes em um sistema completo de recomendação sequencial.

## Mapeamento para o Desafio Final

| Notebook | Competência construída | Quando usar no Desafio Final |
|---|---|---|
| NB01 — Bellman e DP | Policy Iteration, Value Iteration, Equação de Bellman | Ambiente com modelo completo e espaço de estados discreto pequeno |
| NB02 — MC, TD, Q-Learning | Aprendizado model-free; bootstrapping; estimativa de Q(s,a) | Ambiente sem modelo; base para todos os algoritmos desta aula |
| NB03 — SARSA | On-policy vs off-policy; custo de exploração internalizado | Quando penalidades por exploração devem ser consideradas na política |
| NB04 — DQN | Estado contínuo; replay buffer; target network; tríade mortal | Qualquer ambiente com observação contínua e ações discretas |
| NB05 — Comparativo | Escolha do método adequado ao cenário | Diagnóstico inicial do Desafio Final: tabular ou aproximação por função? |

**Guia de decisão rápido:** modelo disponível → DP. Sem modelo, estado discreto pequeno → Q-Learning/SARSA. Estado contínuo, ações discretas → DQN. Ações contínuas → Aula 03.

## Glossário — termos introduzidos nesta aula

| Termo (EN) | Tradução (PT) | Descrição |
|---|---|---|
| model-based | baseado em modelo | Usa um modelo explícito do ambiente (transições e recompensas) para planejar. |
| model-free | livre de modelo | Aprende apenas por interação com o ambiente, sem conhecer as transições. |
| bootstrapping | bootstrapping | Atualizar uma estimativa usando outra estimativa (como V(s') antes do retorno real). |
| on-policy | on-policy | Aprende sobre a política que está sendo executada no momento. |
| off-policy | off-policy | Aprende sobre a política ótima, independentemente da política de comportamento. |

> Glossário completo do curso: [docs/glossario.md](../../docs/glossario.md)

## Leituras e referências

- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2ª ed.). MIT Press. Disponível em: http://incompleteideas.net/book/the-book-2nd.html. Acesso em: abril 2026.
- Brockman, G. et al. (2016). *OpenAI Gym*. arXiv:1606.01540. Disponível em: https://arxiv.org/abs/1606.01540. Acesso em: abril 2026.
- Farama Foundation. *Gymnasium documentation*. Disponível em: https://gymnasium.farama.org. Acesso em: abril 2026.
- Mnih, V. et al. (2015). Human-level control through deep reinforcement learning. *Nature*, 518, 529–533.